# 거래 가능 코스피 종목 30년치 수정주가 추출
- 종목 목록: `tradable_assets.md`의 **코스피 (KRW)** 섹션
- 데이터: `/Volumes/USB_128GB/DataGuide/Kospi/Kospi_수정주가` (DataGuide, 보통주 10년 단위 파일)
- 기간: 최근 30년 (1996-01-01 ~ 데이터 마지막 날짜)
- 수정주가 = 액면분할·감자·증자를 현재 기준으로 소급 조정한 가격 → **수익률 계산용**
- 조회 시점(2026-10-18) 기준이므로 나중에 받은 수정주가 파일과 섞지 말 것

In [ ]:
import re
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/Users/kasia/Desktop/학교/26-2/금프2/프로젝트")
MD_PATH = PROJECT_DIR / "tradable_assets.md"
DATA_DIR = Path("/Volumes/USB_128GB/DataGuide/Kospi/Kospi_수정주가")

START = pd.Timestamp("1996-01-01")
OUT_CSV = PROJECT_DIR / "kospi_adj_close_30y.csv"

try:
    import python_calamine  # noqa: F401  (있으면 엑셀 읽기가 훨씬 빠름)
    ENGINE = "calamine"
except ImportError:
    ENGINE = "openpyxl"
ENGINE

## 1. md 파일에서 코스피 종목코드 추출

In [ ]:
text = MD_PATH.read_text(encoding="utf-8")

# '## 코스피' 섹션만 잘라내기 (다음 '## ' 헤더 또는 문서 끝까지)
sec = re.search(r"^## 코스피.*?$(.*?)(?=^## |\Z)", text, flags=re.M | re.S).group(1)

# '000100 유한양행 / 000150 두산 / ...' 형태 파싱
targets = {}
for item in sec.split("/"):
    m = re.search(r"(\d{6})\s+([^/\n]+)", item)
    if m:
        targets["A" + m.group(1)] = m.group(2).strip()

print(len(targets), "종목")
list(targets.items())[:5]

## 2. 30년 구간에 해당하는 보통주 수정주가 파일 읽기

In [ ]:
def file_range(p):
    s, e = re.search(r"_(\d{8})-(\d{8})", p.name).groups()
    return pd.Timestamp(s), pd.Timestamp(e)

import unicodedata
# 외장 USB 파일명의 한글이 NFD(자모 분리)로 저장돼 있어 NFC로 정규화 후 비교
files = sorted(p for p in DATA_DIR.iterdir()
               if unicodedata.normalize("NFC", p.name).startswith("adj_close(보)_")
               and p.suffix == ".xlsx" and not p.name.startswith("._"))
files = [p for p in files if file_range(p)[1] >= START]
[p.name for p in files]

In [ ]:
def read_dataguide(path):
    raw = pd.read_excel(path, header=None, engine=ENGINE)
    codes = raw.iloc[8, 1:].tolist()                       # 9행: 종목코드
    names = dict(zip(codes, raw.iloc[9, 1:].tolist()))     # 10행: 종목명
    df = raw.iloc[14:].set_index(0)                        # 15행~: 데이터
    df.columns = codes
    df.index = pd.to_datetime(df.index)
    df.index.name = "date"
    df = df.apply(pd.to_numeric, errors="coerce")
    return df, names

frames, dg_names = [], {}
for p in files:
    df, names = read_dataguide(p)
    frames.append(df)
    dg_names.update(names)
    print(f"{p.name}: {df.index.min().date()} ~ {df.index.max().date()}, {df.shape}")

## 3. 이어 붙이기 → 중복 날짜 제거 → 종목 필터

In [ ]:
close_all = pd.concat(frames).sort_index()
close_all = close_all[~close_all.index.duplicated(keep="last")]   # 각 파일 첫 행 = 직전 기간 마지막 거래일
close_all = close_all.loc[START:]

found   = [c for c in targets if c in close_all.columns]
missing = [c for c in targets if c not in close_all.columns]

close = close_all[found]
print("기간:", close.index.min().date(), "~", close.index.max().date(), "/", len(close), "거래일")
print("찾은 종목:", len(found), "/", len(targets))
print("데이터에 없는 종목:", {c: targets[c] for c in missing})

데이터에 없는 종목은 DataGuide 코드 포트폴리오가 **`코스피_일반기업_보통주`**라서 빠진 것들임
(ETF: KODEX 국고채3년·ACE 국고채10년·RISE 국고채30년, 인프라펀드: 맥쿼리인프라 등). 별도로 받아야 함.

## 4. 종목별 데이터 커버리지 확인

In [ ]:
summary = pd.DataFrame({
    "md_name": [targets[c] for c in found],
    "dg_name": [dg_names.get(c) for c in found],
    "first_date": [close[c].first_valid_index() for c in found],
    "last_date":  [close[c].last_valid_index() for c in found],
    "n_obs": close.notna().sum().values,
    "n_zero": (close == 0).sum().values,
}, index=pd.Index(found, name="code"))
summary["full_30y"] = summary["first_date"] <= close.index[0]
print("30년 전체 데이터 보유:", summary["full_30y"].sum(), "종목")
summary.sort_values("first_date")

## 4-1. 검증: 원주가(종가)와 비교
- 마지막 날짜에는 수정주가 = 원주가여야 함
- 삼성전자 2018-05-04 50:1 액면분할 전후로 수정주가는 끊김 없어야 함

In [ ]:
raw_csv = PROJECT_DIR / "kospi_close_30y.csv"   # kospi_close_30y.ipynb 결과물
if raw_csv.exists():
    raw_close = pd.read_csv(raw_csv, index_col=0, parse_dates=True)
    last = close.index[-1]
    diff = (close.loc[last] - raw_close.loc[last, close.columns]).abs()
    print(f"{last.date()} 수정주가 ≠ 종가 종목 수:", (diff > 0).sum())
    print(pd.DataFrame({"종가": raw_close.loc["2018-05-02":"2018-05-08", "A005930"],
                        "수정주가": close.loc["2018-05-02":"2018-05-08", "A005930"]}))

## 5. 저장

In [ ]:
close.to_csv(OUT_CSV, encoding="utf-8-sig")                         # 열 = 종목코드(A######)
summary.to_csv(PROJECT_DIR / "kospi_adj_close_30y_summary.csv", encoding="utf-8-sig")
print("saved:", OUT_CSV, close.shape)
close.tail()